In [1]:
from utils import *


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

In [2]:
n_sim = 100
k_nn = 10
true_set = {0, 1, 2}

In [3]:
settings_list   = ["setting1", "setting2", "setting3", "setting4"]
statistics_list = ["linear", "kfoci", "MDCSIS", "BcorSIS", "Kfilter"]
p_list          = [10, 25]
n_list          = [300]

In [4]:
np.random.seed(0)
df = run_all_experiments(
    settings_list, statistics_list, p_list, n_list,
    true_set=true_set, n_sim=n_sim, k_nn=k_nn
)
df

,setting,statistic,p,n,exact_recovery_rate,containment_rate,avg_selected_size
0,setting1,linear,10,300,0.99,0.99,2.99
1,setting1,kfoci,10,300,1.00,1.00,3.00
2,setting1,MDCSIS,10,300,0.16,0.16,3.00
3,setting1,BcorSIS,10,300,0.95,0.95,3.00
4,setting1,Kfilter,10,300,0.79,0.79,3.00
5,setting1,linear,25,300,0.99,1.00,3.01
6,setting1,kfoci,25,300,1.00,1.00,3.00
7,setting1,MDCSIS,25,300,0.08,0.08,3.00
8,setting1,BcorSIS,25,300,0.87,0.87,3.00
9,setting1,Kfilter,25,300,0.65,0.65,3.00


In [5]:
def format_metrics(sub_df):
    if sub_df.empty:
        return "---"
    exact   = sub_df["exact_recovery_rate"].mean()
    contain = sub_df["containment_rate"].mean()
    size    = sub_df["avg_selected_size"].mean()
    return f"{exact:.2f}/{contain:.2f}/{size:.2f}"

unique_settings = df["setting"].unique()
unique_ps       = df["p"].unique()
methods         = df["statistic"].unique()

pivoted_rows = []
for setting in unique_settings:
    for p in unique_ps:
        row = {"setting": setting, "p": p}
        combo_data = df[(df["setting"] == setting) & (df["p"] == p)]
        if not combo_data.empty:
            for method in methods:
                row[method] = format_metrics(combo_data[combo_data["statistic"] == method])
            pivoted_rows.append(row)

table_df = pd.DataFrame(pivoted_rows).set_index(["setting", "p"])

num_methods   = len(methods)
column_format = "|l|l|" + "c|" * num_methods

latex_output = table_df.to_latex(
    column_format=column_format,
    escape=False,
    multicolumn=True,
    multirow=True,
    index_names=False,
)
latex_output = (
    latex_output
    .replace("\\toprule",    "\\hline")
    .replace("\\midrule",    "\\hline")
    .replace("\\bottomrule", "\\hline")
)
print(latex_output)

\begin{tabular}{|l|l|c|c|c|c|c|}
\hline
 &  & linear & kfoci & MDCSIS & BcorSIS & Kfilter \\
\hline
\multirow[t]{2}{*}{setting1} & 10 & 0.99/0.99/2.99 & 1.00/1.00/3.00 & 0.16/0.16/3.00 & 0.95/0.95/3.00 & 0.79/0.79/3.00 \\
 & 25 & 0.99/1.00/3.01 & 1.00/1.00/3.00 & 0.08/0.08/3.00 & 0.87/0.87/3.00 & 0.65/0.65/3.00 \\
\cline{1-7}
\multirow[t]{2}{*}{setting2} & 10 & 0.86/0.94/3.03 & 0.66/0.85/3.14 & 0.15/0.15/3.00 & 0.23/0.23/3.00 & 0.22/0.22/3.00 \\
 & 25 & 0.76/0.92/3.11 & 0.51/0.75/3.26 & 0.04/0.04/3.00 & 0.13/0.13/3.00 & 0.04/0.04/3.00 \\
\cline{1-7}
\multirow[t]{2}{*}{setting3} & 10 & 0.72/1.00/3.31 & 0.60/0.99/3.45 & 0.16/0.16/3.00 & 0.19/0.19/3.00 & 0.10/0.10/3.00 \\
 & 25 & 0.45/0.97/3.69 & 0.45/0.91/3.59 & 0.03/0.03/3.00 & 0.01/0.01/3.00 & 0.07/0.07/3.00 \\
\cline{1-7}
\multirow[t]{2}{*}{setting4} & 10 & 0.90/0.95/3.01 & 0.98/1.00/3.02 & 0.54/0.54/3.00 & 0.83/0.83/3.00 & 0.69/0.69/3.00 \\
 & 25 & 0.73/0.89/3.12 & 0.84/1.00/3.16 & 0.40/0.40/3.00 & 0.74/0.74/3.00 & 0.60/0.60/3.00 \\
